In [12]:
import pandas as pd
import numpy as np
import os
working_path = os.getcwd()

In [13]:
def start_stop_schedule(start, stop):
# start = '08:00'
# stop = '09:45'
    start_int = int(start[0:2])*60 + int(start[-2:])
    stop_int = int(stop[0:2])*60 + int(stop[-2:])
    
    schedule_list = [0]*start_int + [1]*(stop_int-start_int) + [0]*(24*60 - stop_int)
    
    return schedule_list

def process_NHTS(HOUSEID):

# calculate building attributes
#     HOUSEID = 30000041

    NHTS_EV_df = pd.read_csv(working_path + '/data/NHTS_EV.csv', low_memory=False)

    house_attribute_list = ['HOUSEID', # Household Identifier
                            'HHFAMINC', # Household Income
                            'HHVEHCNT', # Count of household vehicles
                            'HHSIZE', # Count of household members
                            'HHSTATE', # Household State
                            'HH_RACE', # Race of household respondent
                            'HH_HISP', # Hispanic status of household respondent
                            'HH_CBSA', # Core Based Statistical Area (CBSA) FIPS code for the respondent's home address
                            'HBHUR',] # Urban / Rural indicator - Block group

    # more_attribute_list = [] # to be discussed with Alyssa and saquib

    people_attribute_list = ['PERSONID', # Person Identifier
                             'R_AGE_IMP', # Age (imputed)
                             'R_SEX_IMP', # Gender (imputed)
                             'DRIVER',] # Driver status, derived]

    building_dictionary = NHTS_EV_df.loc[NHTS_EV_df.HOUSEID == HOUSEID][house_attribute_list].iloc[0,:].to_dict()

    people_df = NHTS_EV_df.loc[NHTS_EV_df.HOUSEID == HOUSEID][people_attribute_list].set_index(['PERSONID'])


    people_dictionary = {index: row.to_dict() for index, row in people_df.iterrows()}

    building_dictionary['People'] = people_dictionary
    
    
    # calculate schedule array
    start_time_df = NHTS_EV_df.loc[NHTS_EV_df.HOUSEID == HOUSEID].filter(like='STRTTIME_TRIP')
    end_time_df = NHTS_EV_df.loc[NHTS_EV_df.HOUSEID == HOUSEID].filter(like='ENDTIME_TRIP')

    schedule_array_sum = 0
    for (index_start, row_start), (index_end, row_end) in zip(start_time_df.iterrows(), end_time_df.iterrows()):
#         print(index_start, index_end)
        schedule_array = 0
        for (start, stop) in zip(row_start.dropna(), row_end.dropna()):
#             print(start, stop)
            schedule_temp_array = np.array(start_stop_schedule(start, stop))
            schedule_array = schedule_array + schedule_temp_array
        schedule_array = -schedule_array + 1
        
        schedule_array_sum = schedule_array_sum + schedule_array
    
    schedule_array_sum = pd.DataFrame([schedule_array_sum], index = ['occupancy']).T

    return building_dictionary, schedule_array_sum

In [14]:
# process NHTS EV data
NHTS_EV_df = pd.read_csv(working_path + '/data/NHTS_EV.csv',low_memory=False)

HOUSEID_1 = 30000041
HOUSEID_2 = 30001124

HOUSEID_1_results = process_NHTS(HOUSEID_1)
HOUSEID_2_results = process_NHTS(HOUSEID_2)

HOUSEID_1_info = HOUSEID_1_results[0]
HOUSEID_2_info = HOUSEID_2_results[0]

# occupancy
HOUSEID_1_occupancy = HOUSEID_1_results[1]
HOUSEID_2_occupancy = HOUSEID_2_results[1]

# hourly
HOUSEID_1_occupancy_hourly = HOUSEID_1_occupancy.groupby(HOUSEID_1_occupancy.index // 60).mean()
HOUSEID_2_occupancy_hourly = HOUSEID_2_occupancy.groupby(HOUSEID_2_occupancy.index // 60).mean()

# repeat data for days
HOUSEID_1_occupancy_hourly = pd.concat([HOUSEID_1_occupancy_hourly]*365, ignore_index=True)
HOUSEID_2_occupancy_hourly = pd.concat([HOUSEID_2_occupancy_hourly]*365, ignore_index=True)

# save
HOUSEID_1_occupancy_hourly.to_csv('data/HOUSEID_1_occupancy_hourly.csv', index = None)
HOUSEID_2_occupancy_hourly.to_csv('data/HOUSEID_2_occupancy_hourly.csv', index = None)